# Kovasznay Flow Model

## Setup


In [7]:
import sys
from dataclasses import asdict, replace
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
sys.path.append(str(ROOT / "src"))

import torch

from models import (
    MLP, MLPConfig
)
from train import (
    TrainConfig, OptimizerConfig, run_experiment
)
from problems.kovasznay import (
    KovasznayProblem, KovasznayConfig, Domain2D
)
from utils import (
    get_device, make_experiment_dir, make_run_dir, make_sweep_dir,
    rebuild_sweep_summary_csv, save_manifest
)


## Problem Factory


In [8]:
def make_problem(Re: float = 40.0, lambda_bc: float = 1.0, seed: int = 0):
    return KovasznayProblem(
        domain=Domain2D(),
        cfg=KovasznayConfig(Re=Re, lambda_bc=lambda_bc),
        seed=seed,
    )

## Base Config


In [ ]:
seeds = [5246] #[5246, 16567, 19943, 29393, 30373]   # list of seeds (even if only one seed, we need a list)
mlp_layer_dims = (2, 50, 50, 50, 50, 3)
device_preference = "cuda"  # cpu, cuda, or auto
dtype = torch.float64
results_root = ROOT / "runs"
sweep_name = "v6_rtx4090"

In [ ]:
device = get_device(device_preference)
problem_name = KovasznayProblem.name
experiment_index = {}

# Define neural network
model_cfg = MLPConfig(
    layer_dims=mlp_layer_dims,
    activation="tanh",
    init="xavier_uniform",  # also known as Glorot uniform
)
num_params = sum(p.numel() for p in MLP(model_cfg).parameters())

# Instantiate basic Trainer configuration
#  - seed is injected later
#  - can be changed later using 'replace'
train_cfg = TrainConfig(
    steps=100_000,           # max. steps within time constraint 
    time_constraint_s=12.0*60.0,
    log_every=100,
    dtype=dtype,
    nx_int=51, ny_int=51,    # 51*51 = 2601 grid points
    n_bnd=100,               # 4*100 = 400 boundary points
    nx_val=170, ny_val=153,
)

print("MLP layers:", model_cfg.layer_dims)
print("parameters:", num_params)
print("problem:", problem_name)
print("device:", device)
print("dtype:", dtype)
print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("cuda:", torch.cuda.get_device_name(0))

MLP layers: (2, 50, 50, 50, 50, 3)
parameters: 7953
problem: kovasznay
device: cuda
dtype: torch.float64
torch: 2.11.0+cu128
cuda: NVIDIA GeForce RTX 5060 Laptop GPU


## Optimizer Configs


In [11]:
adam_opt_cfg = OptimizerConfig(
    name="adam",
    lr=1e-3,
    cosine_annealing=True,  # until train_cgf.steps
)

lbfgs_opt_cfg = OptimizerConfig(
    name="lbfgs",
    lr=1.0,
    kwargs=dict(
        max_iter=20,
        history_size=100,
        line_search_fn="strong_wolfe",
        tolerance_grad=1e-7,
        tolerance_change=1e-9,
    ),
)

gnng_dense_opt_cfg = OptimizerConfig(
    name="gauss_newton",
    lr=1.0,
    kwargs=dict(
        backend="dense",
        damping_cap=1e-3,
        damping_floor=1e-9,
        damping_scale=1e-3,
        do_line_search=True,
        line_search_steps=31,
    ),
)

gnng_matfree_opt_cfg = OptimizerConfig(
    name="gauss_newton",
    lr=1.0,
    kwargs=dict(
        backend="matfree",
        use_cg_precond=True,
        precond_nx_int=20,
        precond_ny_int=20,
        precond_n_bnd=25,   
        damping_cap=1e-4,
        damping_floor=1e-7,
        damping_scale=1e-4,
        cg_rtol=1e-3,
        cg_atol=1e-7,
        cg_maxiter=100,
        do_line_search=True,
        line_search_steps=31,
    ),
)

## Execution
Sweep over the defined experiments and seeds.

In [ ]:
experiment_specs = [
    dict(
        experiment_name="adam", 
        opt_cfg=adam_opt_cfg,
        train_cfg=replace(train_cfg, log_every=200),
    ),
    dict(
        experiment_name="lbfgs", 
        opt_cfg=lbfgs_opt_cfg, 
        train_cfg=replace(train_cfg, log_every=5)
    ),
    dict(
        experiment_name="gnng_dense_v0",
        opt_cfg=gnng_dense_opt_cfg, 
        train_cfg=replace(train_cfg, log_every=1)
    ),
    dict(
        experiment_name="gnng_dense_v1",
        opt_cfg=replace(
            gnng_dense_opt_cfg, 
            kwargs=dict(
                gnng_dense_opt_cfg.kwargs,
                damping_cap=1e-5,
                damping_floor=1e-16,
                damping_scale=1.0,
            ),
        ), 
        train_cfg=replace(train_cfg, log_every=1)
    ),
    dict(
        experiment_name="gnng_matfree_v0", 
        opt_cfg=gnng_matfree_opt_cfg, 
        train_cfg=replace(train_cfg, log_every=3)
    ),
    dict(
        experiment_name="gnng_matfree_v1", 
        opt_cfg=replace(
            gnng_matfree_opt_cfg, 
            kwargs=dict(
                gnng_matfree_opt_cfg.kwargs,
                damping_cap=1e-5,
                damping_floor=1e-16,
                damping_scale=1.0,
                cg_atol=0.0,
                cg_rtol=1e-2,
            ),
        ), 
        train_cfg=replace(train_cfg, log_every=3)
    ),
    dict(
        experiment_name="gnng_matfree_v2", 
        opt_cfg=replace(
            gnng_matfree_opt_cfg, 
            kwargs=dict(
                gnng_matfree_opt_cfg.kwargs,
                use_cg_precond=False,
                cg_maxiter=200,
            ),
        ), 
        train_cfg=replace(train_cfg, log_every=3)
    ),
]

selected_experiment_names = [
    "adam",
    "lbfgs",
    "gnng_dense_v0",
    "gnng_dense_v1",
    "gnng_matfree_v0",
    "gnng_matfree_v1",
    "gnng_matfree_v2",
]

specs_by_name = {spec["experiment_name"]: spec for spec in experiment_specs}

missing = [name for name in selected_experiment_names if name not in specs_by_name]
if missing:
    raise ValueError(f"Unknown experiment name(s): {missing}")

selected_specs = [specs_by_name[name] for name in selected_experiment_names]

sweep_dir = make_sweep_dir(results_root, problem_name, sweep_name=sweep_name)
save_manifest(
    sweep_dir,
    problem_name=problem_name,
    seeds=seeds,
    experiment_names=selected_experiment_names,
    shared_config={
        "model": asdict(model_cfg),
        "train": asdict(train_cfg),
        "dtype": str(dtype),
        "device": str(device),
    },
)

for spec in selected_specs:
    experiment_dir = make_experiment_dir(sweep_dir, spec["experiment_name"])

    for seed in seeds:
        run_dir = make_run_dir(experiment_dir, seed)
        problem = make_problem(seed=seed)

        history, summary_payload, save_path = run_experiment(
            experiment_name=spec["experiment_name"],
            problem=problem,
            opt_cfg=spec["opt_cfg"],
            train_cfg=replace(spec["train_cfg"], seed=seed),
            model_cfg=model_cfg,
            seed=seed,
            dtype=dtype,
            device=device,
            sweep_name=sweep_name,
            repo_root=ROOT,
            run_dir=run_dir,
            verbose=True,
        )

        rebuild_sweep_summary_csv(sweep_dir)


{'step': '000001', 'experiment': 'gnng_matfree_v0', 'wall time [s]': '     2.2', 'loss': '6.01e-01', 'rel L2 (mean)': '1.74e+00', 'backend': 'matfree', 'damping': '1.00e-04', 'CG iter': '003', 'step size': '3.98e-01', 'lr': '1.00e+00', 'gpu_peak_mem_mb': '1489.1'}
{'step': '000003', 'experiment': 'gnng_matfree_v0', 'wall time [s]': '     6.7', 'loss': '3.40e-01', 'rel L2 (mean)': '1.19e+00', 'backend': 'matfree', 'damping': '4.04e-05', 'CG iter': '004', 'step size': '6.31e-02', 'lr': '1.00e+00', 'gpu_peak_mem_mb': '1489.1'}
{'step': '000006', 'experiment': 'gnng_matfree_v0', 'wall time [s]': '    13.4', 'loss': '2.28e-01', 'rel L2 (mean)': '9.87e-01', 'backend': 'matfree', 'damping': '2.30e-05', 'CG iter': '004', 'step size': '1.00e-02', 'lr': '1.00e+00', 'gpu_peak_mem_mb': '1489.1'}
{'step': '000009', 'experiment': 'gnng_matfree_v0', 'wall time [s]': '    20.0', 'loss': '2.17e-01', 'rel L2 (mean)': '9.47e-01', 'backend': 'matfree', 'damping': '2.23e-05', 'CG iter': '004', 'step size':